In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, NumericType
import pandas as pd
import numpy as np
from pyspark.testing import assertSchemaEqual
pd.set_option('display.max_colwidth', None)

def compare_dataframes(
    df_base,
    df_target,
    cols_to_compare,
    cols_keys,
    threshold=1e-6,
    suffix_base="_base",
    suffix_target="_target"
):
    def suffix_cols(cols, suffix):
        return [F.col(c).alias(c + suffix) if c not in cols_keys else F.col(c) for c in cols]

    # Compare number of records in each dataframe
    df_base_count = df_base.count()
    df_target_count = df_target.count()
    if df_base_count != df_target_count:
        raise ValueError(f"Count mismatch: df_base={df_base_count}, df_target={df_target_count}")

    # Select & rename
    df_b = df_base.select(*suffix_cols(cols_to_compare + cols_keys, suffix_base))
    df_t = df_target.select(*suffix_cols(cols_to_compare + cols_keys, suffix_target))

    # Join
    df = df_b.join(df_t, cols_keys, "outer")

    # Build comparison expressions
    expr = df.columns.copy()

    for c in cols_to_compare:
        l = df.col(c + suffix_base)
        r = df.col(c + suffix_target)

        ldt = df.select(l).schema[0].dataType
        rdt = df.select(r).schema[0].dataType

        if isinstance(ldt, StringType) and isinstance(rdt, StringType):
            expr.append(
                F.when(l.isNull() & r.isNull(), 1)
                .when(l == r, 1)
                .otherwise(0)
                .alias(c + "_eq")
            )
        elif isinstance(ldt, NumericType) and isinstance(rdt, NumericType):
            expr.extend([
                (l - r).alias(c + "_diff"),
                F.when(l.isNull() & r.isNull(), 1)
                .when(F.abs(l - r) <= threshold, 1)
                .otherwise(0)
                .alias(c + "_eq")
            ])

    df_cmp = df.select(*expr).cache()
    # df_cmp.count()

    # Summary
    eq_cols = [c for c in df_cmp.columns if c.endswith("_eq")]

    ones = df_cmp.agg(*[F.sum(c).alias(c) for c in eq_cols]).withColumn("eq", F.lit(1))
    zeros = df_cmp.agg(*[(F.count("*") - F.sum(c)).alias(c) for c in eq_cols]).withColumn("eq", F.lit(0))

    df_summary = ones.unionByName(zeros).select("eq", *eq_cols)

    pd_summary = (
        df_summary
        .orderBy("eq")
        .toPandas()
        .set_index("eq")
        .T
        .reset_index()
        .rename(columns={"index": "var", 0: "count_0", 1: "count_1"})
        .sort_values("count_0", ascending=False)
    )

    pd_summary['porc_equiv'] = np.divide(pd_summary['count_1'], pd_summary['count_0'] + pd_summary['count_1'])*100
    pd_summary['porc_equiv'] = pd_summary['porc_equiv'].round(6)
    pd_summary['validacion'] = np.where(pd_summary['porc_equiv'] >= 99.0, 'ok', 'falla')

    return df_cmp, pd_summary


from pyspark.sql.types import StructType

def compare_schemas(schema1: StructType, schema2: StructType):
    fields1 = {f.name: f for f in schema1.fields}
    fields2 = {f.name: f for f in schema2.fields}
    all_cols = set(fields1.keys()).union(fields2.keys())

    diffs = []
    for col in sorted(all_cols):
        f1 = fields1.get(col)
        f2 = fields2.get(col)

        if f1 is None:
            diffs.append(f"+ Column only in target: {col} {f2.dataType}")
        elif f2 is None:
            diffs.append(f"- Column only in base: {col} {f1.dataType}")
        else:
            differences = []
            if f1.dataType != f2.dataType:
                differences.append(f"type: {f1.dataType} != {f2.dataType}")
            if f1.nullable != f2.nullable:
                differences.append(f"nullable: {f1.nullable} != {f2.nullable}")

            if differences:
                diffs.append(f"* Column '{col}' differs -> " + ", ".join(differences))

    return diffs

In [0]:
df_base = spark.table("catalog_cemm_expl_bcp_prod.bcp_expl_007.bcp_mt_troncal_bhv_pyme_v1")
df_target = spark.table("catalog_lhcl_prod_bcp_expl.bcp_edv_mmgr.t06086_pd_bhv_cliente_pyme_202306_202604_32var")

# compare_schemas(df_base.schema, df_target.schema)

config_codmes_list = [202604]

cols_keys = ['codmes', 'codclaveunicocli']
vars_cols = [c for c in df_base.columns if c not in cols_keys + ["codclavepartycli", "codinternocomputacional"]]
for codmes in config_codmes_list:
    df_base_codmes = df_base.filter(F.col('codmes')==F.lit(codmes))
    df_target_codmes = df_target.filter(F.col('codmes')==F.lit(codmes))
    df_codmes, summary_codmes = compare_dataframes(
        df_base=df_base_codmes,
        df_target=df_target_codmes,
        cols_to_compare=vars_cols,
        cols_keys=cols_keys,
        threshold=1e-6,
        suffix_base="_base",
        suffix_target="_target"
    )
    print('='*60)
    print(codmes)
    print('-'*60)
    display(summary_codmes)